In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

def run_inference_on_sample(npz_file_path, model, sequence_length=450, fs=30.0, pad_margin=30):
    """
    Execute inference on an NPZ sample, applying reflection padding prior to 
    bandpass filtering to completely suppress convolution boundary ringing at frame 0.
    """
    print(f"--- INFERENCE RUN ON: {os.path.basename(npz_file_path)} ---")
    data = load_npz_file(npz_file_path)
    if data is None:
        return None, None
    
    ppg_key = 'ppg_values' if 'ppg_values' in data else 'ppg'
    if len(data[ppg_key]) < sequence_length:
        print("Error: Input file length is shorter than required sequence length.")
        return None, None
        
    # 1. Ground Truth Pipeline (Padded filtering to prevent edge artifacts)
    raw_target_ppg = data[ppg_key][:sequence_length].copy()
    gt_padded = np.pad(raw_target_ppg, pad_margin, mode='reflect')
    gt_filtered = butter_bandpass_filter(gt_padded - np.mean(gt_padded), fs=fs)
    gt_cropped = gt_filtered[pad_margin:-pad_margin]
    gt_norm = (gt_cropped - np.mean(gt_cropped)) / (np.std(gt_cropped) + 1e-8)
    
    # 2. Multi-ROI Input Tensor Preparation
    roi_inputs = []
    for roi_name in ROI_ORDER:
        actual_key = next((k for k in data.files if roi_name == k), None)
        if actual_key is not None:
            roi_data = data[actual_key][:sequence_length]
        else:
            roi_data = np.zeros((sequence_length, 24, 24, 3), dtype=np.uint8)
        roi_inputs.append(roi_data.astype(np.float32) / 255.0)
        
    stacked_rois = np.stack(roi_inputs, axis=0)
    stacked_rois = np.transpose(stacked_rois, (0, 4, 1, 2, 3))
    stacked_rois = np.reshape(stacked_rois, (30, sequence_length, 24, 24))
    
    input_tensor = torch.tensor(stacked_rois, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    
    # 3. Network Forward Pass
    model.eval()
    with torch.no_grad():
        output_signal = model(input_tensor).cpu().numpy()[0]
        
    # 4. Prediction Pipeline: Pad -> Filter -> Crop -> Z-Score Normalize
    pred_padded = np.pad(output_signal, pad_margin, mode='reflect')
    pred_filtered = butter_bandpass_filter(pred_padded - np.mean(pred_padded), fs=fs)
    pred_cropped = pred_filtered[pad_margin:-pad_margin]
    pred_norm = (pred_cropped - np.mean(pred_cropped)) / (np.std(pred_cropped) + 1e-8)
    
    # 5. Heart Rate Metrics Calculation
    hr_pred = calculate_bpm_from_fft(pred_norm, fs=fs)
    hr_true = calculate_bpm_from_fft(gt_norm, fs=fs)
    
    print(f"  Predicted HR  : {hr_pred:.2f} BPM")
    print(f"  Ground-Truth  : {hr_true:.2f} BPM")
    print(f"  Absolute Error: {abs(hr_pred - hr_true):.2f} BPM")
    print("----------------------------------------------------\n")
    return pred_norm, gt_norm

In [ ]:
# ============================================================================
# EXECUTION & DASHBOARD PLOTTING CELL
# ============================================================================

# Select sample file from validation split
sample_val_file = val_split[0] if len(val_split) > 0 else npz_files[0]
inference_pred, inference_gt = run_inference_on_sample(sample_val_file, model)

# Create 2x2 Visualization Dashboard
fig, axs = plt.subplots(2, 2, figsize=(15, 11))

# --- Subplot 1: Convergence Curves (DUAL AXIS) ---
epochs_range = range(1, len(train_losses) + 1)
ax1 = axs[0, 0]
line1 = ax1.plot(epochs_range, train_losses, label='Train Loss', color='#1f77b4', linewidth=2.5, marker='o', markersize=3)
line2 = ax1.plot(epochs_range, val_losses, label='Val Loss', color='#ff7f0e', linewidth=2.5, marker='s', markersize=3)
ax1.set_title('Model Convergence (Loss vs. Validation PCC)')
ax1.set_xlabel('Training Epochs')
ax1.set_ylabel('Tri-Objective Loss (Dynamic Weights)', color='#333333')
ax1.tick_params(axis='y', labelcolor='#333333')

# Add PCC on secondary Y axis
ax2 = ax1.twinx()
line3 = ax2.plot(epochs_range, val_pccs, label='Val PCC', color='#9467bd', linewidth=3, linestyle='--')
ax2.set_ylabel('Pearson Correlation Coefficient (Higher is Better)', color='#9467bd', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#9467bd')

# Combine legends from both axes
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# --- Subplot 2: Standardized Waveform Alignment ---
if inference_pred is not None and inference_gt is not None:
    axs[0, 1].plot(inference_gt, label='GT PPG (Normalized μ=0, σ=1)', color='#7f7f7f', alpha=0.85, linewidth=2)
    axs[0, 1].plot(inference_pred, label='10-ROI Predicted rPPG', color='#2ca02c', linewidth=2.2)
    axs[0, 1].set_title('Signal Waveform Comparison (Inference Showcase)')
    axs[0, 1].set_xlabel('Frames Timeline')
    axs[0, 1].set_ylabel('Normalized Amplitude (Z-Score)')
    axs[0, 1].set_ylim(-3.5, 3.5)  # Constrain y-axis to valid standardized bounds
    axs[0, 1].legend(loc='upper right')

# --- Subplot 3: Spectral Density Alignment ---
if inference_pred is not None and inference_gt is not None:
    freqs_p, psd_p = welch(inference_pred, fs=30.0, nperseg=len(inference_pred))
    freqs_t, psd_t = welch(inference_gt, fs=30.0, nperseg=len(inference_gt))
    
    axs[1, 0].plot(freqs_t * 60.0, psd_t, label='GT Spectrum', color='#7f7f7f', alpha=0.85, linewidth=2)
    axs[1, 0].plot(freqs_p * 60.0, psd_p, label='Pred Spectrum', color='#d62728', linewidth=2.2)
    axs[1, 0].set_xlim(40, 180)
    axs[1, 0].set_title('Power Spectral Density (PSD) Comparison')
    axs[1, 0].set_xlabel('Heart Rate (BPM)')
    axs[1, 0].set_ylabel('Spectral Magnitude')
    axs[1, 0].legend(loc='upper right')

# --- Subplot 4: Summary Card ---
axs[1, 1].axis('off')
metric_summary_card = (
    "10-ROI Spatio-Temporal Evaluation Summary\n\n"
    f"\u2022 Pearson Correlation Coefficient (PCC): {metrics['PCC']:.4f}\n"
    f"\u2022 Mean Absolute Error (MAE): {metrics['MAE']:.2f} BPM\n"
    f"\u2022 Root Mean Squared Error (RMSE): {metrics['RMSE']:.2f} BPM\n"
    f"\u2022 Signal-to-Noise Ratio (SNR): {metrics['SNR']:.2f} dB\n\n"
    f"Architecture & Configuration:\n"
    f"\u2022 Spatio-Temporal Conv3D + Deep 1D Temporal Encoder\n"
    f"\u2022 Input Channels: 30 (10 Face Regions \u00d7 3 RGB)\n"
    f"\u2022 Forward Normalization: AC/DC [ (x - \u03bc) / (\u03bc + \u03b5) ]\n"
    f"\u2022 Sequence segment window: 450 frames (15.0s @ 30 FPS)\n"
    f"\u2022 Loss Strategy: Dynamic Weighting (Phase + Freq + Amplitude)"
)
axs[1, 1].text(0.05, 0.5, metric_summary_card, fontsize=11, va='center',
              bbox=dict(boxstyle='round,pad=1.5', facecolor='#f8f9fa', edgecolor='#dee2e6'))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# AVI INFERENCE CELL - MediaPipe 10-ROI -> Deeper Model Inference with GT vs Predicted Plot
# ============================================================================

import os
import cv2
import math
import imageio
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, welch
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ----------------------------------------------------------------------------
# Configuration & Constants
# ----------------------------------------------------------------------------
# Update these paths as needed
VIDEO_PATH = '/path/to/your/video.avi'  # Path to your target .avi video
MEDIAPIPE_MODEL_PATH = '/path/to/face_landmarker.task'
MODEL_CHECKPOINT_PATH = '../models/spatiotemporal_physnet_best.pth'  # or your deeper model path

CHUNK_SIZE = 450
ROI_BOX_SIZE = (24, 24)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Exact 10 ROI ordering expected by the model (30 input channels)
ROI_ORDER = [
    'roi_forehead',
    'roi_left_cheek_280',
    'roi_right_cheek_50',
    'roi_nose',
    'roi_chin',
    'roi_chin_199',
    'roi_left_eye',
    'roi_right_eye',
    'roi_mouth',
    'roi_full_face'
]

# MediaPipe landmark mappings for the 10 ROIs
ROIS_MAPPING = {
    'roi_full_face': list(range(468)),
    'roi_forehead': [10, 67, 69, 108, 109, 151, 337, 338, 297, 299, 9, 8],
    'roi_left_eye': [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246],
    'roi_right_eye': [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398],
    'roi_nose': [1, 2, 98, 327, 328, 2, 4, 5, 195, 197, 6, 168],
    'roi_mouth': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 95],
    'roi_chin': [152, 148, 176, 149, 150, 136, 172, 377, 400, 378, 379, 365, 397],
    'roi_right_cheek_50': [50],
    'roi_left_cheek_280': [280],
    'roi_chin_199': [199]
}

# ----------------------------------------------------------------------------
# MediaPipe Detector Class
# ----------------------------------------------------------------------------
class ModernFaceMeshDetector:
    def __init__(self, model_path):
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            num_faces=1,
            min_face_detection_confidence=0.5,
            min_face_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.landmarker = vision.FaceLandmarker.create_from_options(options)
        self.timestamp_ms = 0

    def detect_landmarks(self, frame_rgb):
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        self.timestamp_ms += 33  # ~30 FPS step
        detection_result = self.landmarker.detect_for_video(mp_image, self.timestamp_ms)
        
        if detection_result.face_landmarks:
            h, w = frame_rgb.shape[:2]
            landmarks = detection_result.face_landmarks[0]
            return np.array([[lm.x * w, lm.y * h] for lm in landmarks], dtype=np.float32)
        return None

    def reset(self):
        self.timestamp_ms = 0

    def close(self):
        self.landmarker.close()

# ----------------------------------------------------------------------------
# ROI Extraction Functions
# ----------------------------------------------------------------------------
def extract_roi_bbox(landmarks, indices, frame_shape, target_size=ROI_BOX_SIZE):
    h, w = frame_shape
    roi_lms = landmarks[indices]
    
    if len(indices) == 1:
        # Point landmark (e.g., cheek, chin point) -> center crop around landmark
        cx, cy = roi_lms[0]
        crop_w, crop_h = target_size
        x = int(cx - crop_w / 2)
        y = int(cy - crop_h / 2)
    else:
        # Multi-point ROI -> calculate bounding box
        min_x, min_y = np.min(roi_lms, axis=0)
        max_x, max_y = np.max(roi_lms, axis=0)
        x, y = int(min_x), int(min_y)
        crop_w, crop_h = int(max_x - min_x), int(max_y - min_y)
        
    x = max(0, min(x, w - 1))
    y = max(0, min(y, h - 1))
    crop_w = max(1, min(crop_w, w - x))
    crop_h = max(1, min(crop_h, h - y))
    
    return (x, y, crop_w, crop_h)

def extract_roi_region(frame, bbox, target_size=ROI_BOX_SIZE):
    x, y, w, h = bbox
    crop = frame[y:y+h, x:x+w]
    if crop.size == 0:
        return np.zeros((*target_size, 3), dtype=np.uint8)
    return cv2.resize(crop, target_size, interpolation=cv2.INTER_AREA)

# ----------------------------------------------------------------------------
# Signal Processing Helper Functions
# ----------------------------------------------------------------------------
def butter_bandpass_filter(data, lowcut=0.75, highcut=2.5, fs=30.0, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def calculate_bpm_from_fft(signal_1d, fs=30.0, lowcut=0.75, highcut=2.5):
    signal_cleaned = signal_1d - np.mean(signal_1d)
    freqs, psd = welch(signal_cleaned, fs=fs, nperseg=len(signal_cleaned))
    valid_idx = np.where((freqs >= lowcut) & (freqs <= highcut))[0]
    if len(valid_idx) == 0:
        return 75.0
    peak_freq = freqs[valid_idx[np.argmax(psd[valid_idx])]]
    return peak_freq * 60.0

# ----------------------------------------------------------------------------
# Video Processing Pipeline for Deeper Model
# ----------------------------------------------------------------------------
def process_avi_for_inference(avi_path, detector, model, chunk_size=CHUNK_SIZE, fs=30.0, pad_margin=30):
    """
    Process AVI video: extract frames, detect landmarks, crop 10 ROIs,
    run inference through deeper model, and return predicted + ground truth signals.
    """
    print(f"--- AVI INFERENCE ON: {os.path.basename(avi_path)} ---")
    
    # 1. Load video frames using imageio
    reader = imageio.get_reader(avi_path, 'ffmpeg')
    meta = reader.get_meta_data()
    video_fps = meta.get('fps', fs)
    
    frames = []
    for i, frame in enumerate(reader):
        if i >= chunk_size:
            break
        frames.append(frame)
    reader.close()
    
    num_frames = len(frames)
    print(f"Loaded {num_frames} frames from '{os.path.basename(avi_path)}' @ {video_fps:.2f} FPS")
    
    # 2. Pad short videos if needed
    if num_frames < chunk_size:
        pad_count = chunk_size - num_frames
        frames += [frames[-1]] * pad_count
        print(f"Padded video from {num_frames} to {chunk_size} frames.")
    
    # 3. Extract landmarks per frame
    chunk_landmarks = []
    detector.reset()
    for frame in frames:
        lms = detector.detect_landmarks(frame)
        if lms is not None:
            chunk_landmarks.append(lms)
        elif chunk_landmarks:
            chunk_landmarks.append(chunk_landmarks[-1].copy())
        else:
            chunk_landmarks.append(np.zeros((468, 2), dtype='float32'))
    chunk_landmarks = np.array(chunk_landmarks)
    
    # 4. Crop ROIs for all frames
    roi_inputs = []
    for roi_name in ROI_ORDER:
        roi_indices = ROIS_MAPPING[roi_name]
        roi_frames = []
        for frame_idx, frame in enumerate(frames):
            landmarks = chunk_landmarks[frame_idx]
            if np.all(landmarks == 0):
                h, w = frame.shape[:2]
                bbox = (w // 2 - ROI_BOX_SIZE[0] // 2, h // 2 - ROI_BOX_SIZE[1] // 2, ROI_BOX_SIZE[0], ROI_BOX_SIZE[1])
            else:
                bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)
            roi_region = extract_roi_region(frame, bbox)
            roi_frames.append(roi_region)
        roi_array = np.array(roi_frames, dtype=np.float32) / 255.0  # (T, 24, 24, 3)
        roi_inputs.append(roi_array)
    
    # 5. Format input tensor: (10, T, 24, 24, 3) -> (1, 30, T, 24, 24)
    stacked_rois = np.stack(roi_inputs, axis=0)                     # (10, T, 24, 24, 3)
    stacked_rois = np.transpose(stacked_rois, (0, 4, 1, 2, 3))     # (10, 3, T, 24, 24)
    stacked_rois = np.reshape(stacked_rois, (30, chunk_size, 24, 24)) # (30, T, 24, 24)
    input_tensor = torch.tensor(stacked_rois, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    
    # 6. Load ground truth PPG from corresponding NPZ file
    npz_path = avi_path.replace('.avi', '.npz')
    if not os.path.exists(npz_path):
        npz_path = avi_path.replace('.avi', '') + '_data.npz'
    
    gt_signal = None
    if os.path.exists(npz_path):
        data = load_npz_file(npz_path)
        if data is not None:
            ppg_key = 'ppg_values' if 'ppg_values' in data else 'ppg'
            raw_gt = data[ppg_key][:chunk_size].copy()
            gt_padded = np.pad(raw_gt, pad_margin, mode='reflect')
            gt_filtered = butter_bandpass_filter(gt_padded - np.mean(gt_padded), fs=video_fps)
            gt_cropped = gt_filtered[pad_margin:-pad_margin]
            gt_signal = (gt_cropped - np.mean(gt_cropped)) / (np.std(gt_cropped) + 1e-8)
            print(f"Loaded ground truth PPG from: {os.path.basename(npz_path)}")
        else:
            print("Warning: Could not load NPZ file")
    else:
        print(f"Warning: No corresponding NPZ file found at: {npz_path}")
    
    # 7. Run inference through model
    model.eval()
    with torch.no_grad():
        output_signal = model(input_tensor).cpu().numpy()[0]
    
    # 8. Process prediction with padding, filtering, cropping, normalization
    pred_padded = np.pad(output_signal, pad_margin, mode='reflect')
    pred_filtered = butter_bandpass_filter(pred_padded - np.mean(pred_padded), fs=video_fps)
    pred_cropped = pred_filtered[pad_margin:-pad_margin]
    pred_signal = (pred_cropped - np.mean(pred_cropped)) / (np.std(pred_cropped) + 1e-8)
    
    # 9. Calculate heart rate metrics
    hr_pred = calculate_bpm_from_fft(pred_signal, fs=video_fps)
    hr_true = calculate_bpm_from_fft(gt_signal, fs=video_fps) if gt_signal is not None else None
    
    print(f"  Predicted HR  : {hr_pred:.2f} BPM")
    if hr_true is not None:
        print(f"  Ground-Truth  : {hr_true:.2f} BPM")
        print(f"  Absolute Error: {abs(hr_pred - hr_true):.2f} BPM")
    print("----------------------------------------------------\n")
    
    return pred_signal, gt_signal, hr_pred, hr_true

# ----------------------------------------------------------------------------
# AVI Inference Execution & Plotting
# ----------------------------------------------------------------------------

# Initialize MediaPipe Detector
print("Initializing MediaPipe Face Landmarker...")
detector = ModernFaceMeshDetector(model_path=MEDIAPIPE_MODEL_PATH)
print("MediaPipe detector initialized.")

# Load your deeper model (replace with your actual model loading code)
# model = YourDeeperModel().to(DEVICE)
# if os.path.exists(MODEL_CHECKPOINT_PATH):
#     checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
#     model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
#     print(f"Loaded model weights from: {MODEL_CHECKPOINT_PATH}")

# Run AVI inference
# Update VIDEO_PATH to your actual video file
avi_pred, avi_gt, pred_hr, true_hr = process_avi_for_inference(VIDEO_PATH, detector, model)

# Plot Ground Truth vs Predicted for AVI
if avi_pred is not None:
    plt.figure(figsize=(16, 8))
    
    # Plot 1: Waveform Comparison
    plt.subplot(2, 1, 1)
    if avi_gt is not None:
        plt.plot(avi_gt, label=f'Ground Truth PPG ({true_hr:.1f} BPM)' if true_hr else 'Ground Truth PPG', 
                 color='#1f77b4', linewidth=2, alpha=0.8)
    plt.plot(avi_pred, label=f'Predicted rPPG ({pred_hr:.1f} BPM)', color='#ff7f0e', linewidth=2)
    plt.title(f'AVI Inference: {os.path.basename(VIDEO_PATH)} - GT vs Predicted Signal', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Frames')
    plt.ylabel('Normalized Amplitude (Z-Score)')
    plt.ylim(-3.5, 3.5)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 2: PSD Comparison
    plt.subplot(2, 1, 2)
    if avi_gt is not None:
        freqs_p, psd_p = welch(avi_pred, fs=30.0, nperseg=len(avi_pred))
        freqs_t, psd_t = welch(avi_gt, fs=30.0, nperseg=len(avi_gt))
        plt.plot(freqs_t * 60.0, psd_t, label='GT Spectrum', color='#1f77b4', alpha=0.8, linewidth=2)
        plt.plot(freqs_p * 60.0, psd_p, label='Predicted Spectrum', color='#ff7f0e', linewidth=2)
        plt.xlim(40, 180)
        plt.title('Power Spectral Density (PSD) Comparison', fontsize=12, fontweight='bold')
        plt.xlabel('Heart Rate (BPM)')
        plt.ylabel('Power Spectral Density')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\n=== AVI Inference Summary ===")
    print(f"Video: {os.path.basename(VIDEO_PATH)}")
    print(f"Predicted HR: {pred_hr:.2f} BPM")
    if true_hr is not None:
        print(f"Ground Truth HR: {true_hr:.2f} BPM")
        print(f"Absolute Error: {abs(pred_hr - true_hr):.2f} BPM")
    print("================================\n")

# Clean up
detector.close()